# OFC 2026 — PI-EDFA + FNO + Residual Learning

Official 95-channel EDFA data | Group-heldout PI baseline vs PI+FNO

**TestGround is post-selection audit ONLY, never use it for training or hyperparameter tuning.**

In [ ]:
import subprocess,sys,torch
from pathlib import Path
print("torch",torch.__version__,"cuda",torch.cuda.is_available())
subprocess.run([sys.executable,"-m","pip","-q","install","numpy","pandas","scikit-learn","matplotlib"],check=True)
subprocess.run(["bash","-lc","git lfs version || (apt-get update -qq && apt-get install -y -qq git-lfs)"],check=True)
def sh(cmd):
    print("+"," ".join(map(str,cmd)),flush=True)
    subprocess.run(list(map(str,cmd)),check=True)

## 1. Get the repository and official Git LFS CSVs

In [ ]:
repo=Path("/content/LSCNS")
if not (repo/".git").exists():
    sh(["git","clone","--depth","1","--filter=blob:none","--sparse",
        "https://github.com/kimheeseo/LSCNS.git",repo])
sh(["git","-C",repo,"sparse-checkout","set",
    "26OFC_ML_challenge/COSMOS_Transfer_Learning",
    "26OFC_ML_challenge/PI_EDFA_FNO_Residual"])
root=Path("/content/ofc-ml-challenge-data-code")
if not (root/".git").exists():
    sh(["bash","-lc",f"GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --filter=blob:none --sparse https://github.com/ofc-ml-challenge/ofc-ml-challenge-data-code.git {root}"])
sh(["git","-C",root,"sparse-checkout","set","Features"])
sh(["git","-C",root,"lfs","install","--local"])

## 2. Training configuration

Defaults run fresh PI training followed by frozen-PI FNO residual training. Optionally pretrain PI on COSMOS.

In [ ]:
COSMOS_PRETRAIN_EPOCHS=0    # 30: optional PI pretrain
COSMOS_MAX_ROWS=4000          # 0 = all COSMOS source rows
PI_EPOCHS=180
FNO_EPOCHS=180
FAST_SMOKE=False              # quick functional check, NOT accuracy
EVALUATE_TESTGROUND=False     # switch on only AFTER model selection
required=["Features/Train/train_features.csv","Features/Train/train_labels.csv",
          "Features/Test/test_features.csv"]
if COSMOS_PRETRAIN_EPOCHS:
    required+=["Features/Train/COSMOS_features.csv","Features/Train/COSMOS_labels.csv"]
if EVALUATE_TESTGROUND:
    required+=["Features/TestGround/test_features.csv","Features/TestGround/test_labels.csv"]
sh(["git","-C",root,"lfs","pull","--include="+",".join(required),"--exclude="])
for item in required:
    f=root/item
    assert f.exists(),f
    with f.open("rb") as stream:head=stream.read(128)
    assert not head.startswith(b"version https://git-lfs"),f"Actual Git LFS CSV missing: {f}"
print("Official CSV files ready")

## 3. Train and compare PI-only with PI+FNO

In [ ]:
runner=repo/"26OFC_ML_challenge/PI_EDFA_FNO_Residual/train.py"
output=Path("/content/fno_results")
command=[sys.executable,str(runner),"--data-root",str(root),"--output",str(output),
         "--pi-epochs",str(2 if FAST_SMOKE else PI_EPOCHS),
         "--fno-epochs",str(2 if FAST_SMOKE else FNO_EPOCHS),
         "--pretrain-epochs",str(COSMOS_PRETRAIN_EPOCHS),
         "--cosmos-max-rows",str(COSMOS_MAX_ROWS)]
if EVALUATE_TESTGROUND:command+=["--evaluate-testground"]
sh(command)

## 4. Print actual metrics, breakdown and learning curves

Figures below appear **only when the previous training cell finishes**. Lower scores are better.

In [ ]:
import json,pandas as pd
from IPython.display import display
r=json.loads((output/"validation_comparison.json").read_text())
print("Recent Train grouped holdout:",r["validation_rows"],"samples")
rows=[{"model":name,"category":cat,**metrics} for name in ("PI_EDFA","PI_EDFA_FNO")
      for cat,metrics in r[name].items()]
display(pd.DataFrame(rows).round(6))
if (output/"testground_comparison.json").exists():
    audit=json.loads((output/"testground_comparison.json").read_text())
    print("Post-selection public TestGround: NOT the hidden Kaggle leaderboard")
    rows=[{"model":model,"category":cat,**v} for model,groups in audit.items()
          for cat,v in groups.items()]
    display(pd.DataFrame(rows).round(6))

In [ ]:
import matplotlib.pyplot as plt
for stage in ("recent_pi","recent_fno"):
    df=pd.read_csv(output/(stage+"_history.csv"))
    plt.plot(df.epoch,df.val_score,label=stage)
plt.xlabel("Epoch");plt.ylabel("Internal validation score");plt.legend()
plt.grid(alpha=.2);plt.show()

## 5. Save outputs

Creates saved checkpoint, validation comparison JSON and 95-channel Kaggle-format submission.

In [ ]:
import shutil
archive=shutil.make_archive("/content/ofc_pi_fno_results","zip",output)
print("Results:",archive)
print("Kaggle-style CSV:",output/"submission.csv")
from google.colab import files
files.download(archive)